In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


hpi = spark.table("scottish_housing_ws.silver.uk_hpi")
lbtt = spark.table("scottish_housing_ws.silver.lbtt_table_1")
festival = spark.table("scottish_housing_ws.silver.festival_dates")

In [0]:
hpi_edinburgh = (
    hpi
    .filter(F.col("area_code") == "S12000036")
    .select("report_date", "year_month", "region_name", "area_code", "average_price", "sales_volume")
)

In [0]:
festival_flags = (
    festival
    .select(
        "year_month",
        "is_festival_month",
        "festival_days_in_month",
        "is_cancelled"
    )
)


In [0]:
lbtt_clean = (
    lbtt
    .select(
        "year_month",
        "is_provisional",
        "residential_returns",
        "residential_tax_total"
    )
)

In [0]:

gold = (
    hpi_edinburgh
    .join(festival_flags, on="year_month", how="left")
    .join(lbtt_clean, on="year_month", how="left")
    .withColumn(
        "is_festival_month",
        F.coalesce(F.col("is_festival_month"), F.lit(False))
    )
    .withColumn(
        "festival_days_in_month",
        F.coalesce(F.col("festival_days_in_month"), F.lit(0))
    )
    .withColumn(
        "is_cancelled",
        F.coalesce(F.col("is_cancelled"), F.lit(False))
    )
    .withColumn(
        "month_number",
        F.month("report_date")
    )
    .select(
        "report_date",
        "year_month",
        "region_name",
        "area_code",
        "average_price",
        "sales_volume",
        "is_festival_month",
        "festival_days_in_month",
        "is_cancelled",
        "month_number",
        "is_provisional",
        "residential_returns",
        "residential_tax_total"
    )
    .orderBy("report_date")
)

In [0]:
print(f"Row count: {gold.count()}")

# Show a festival month and surrounding months for comparison
gold.filter(F.col("year_month").isin("2023-07", "2023-08", "2023-09", "2023-10")).show()

# Show the cancelled festival year
gold.filter(F.col("year_month").isin("2020-07", "2020-08", "2020-09")).show()

In [0]:
(
    gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.gold.festival_month_effect")
)

In [0]:
(
    gold.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv("abfss://data@scottishhousingdl.dfs.core.windows.net/exports/gold_festival_month_effect/")
)

print("Gold 07 complete.")

In [0]:
# Festival Month Effect
# Sources: silver.uk_hpi, silver.lbtt_table_1, silver.festival_dates
# Grain: monthly, City of Edinburgh only
# Join key: year_month
#
# Tests whether the Edinburgh Festival in July/August creates a measurable
# distortion in Edinburgh house prices or transaction volumes.
# festival_dates covers July and August only, 2004 to 2026.
# lbtt_table_1 covers Apr 2015 onwards -- used for tax revenue context.